## Validação da Hipótese de Multinormalidade

Testes formais aplicados ao conjunto de treino completo (subamostra de 5 000 pts) e ao subconjunto LHC (2 000 pts).

| Teste | Estatística | H₀ |
|-------|-------------|-----|
| **Mardia assimetria** | $\hat{\kappa} = \frac{n}{6} b_{1,p} \sim \chi^2_{p(p+1)(p+2)/6}$ | distribuição é simétrica como a normal |
| **Mardia curtose** | $z = (b_{2,p} - p(p+2)) / \sqrt{8p(p+2)/n} \sim \mathcal{N}(0,1)$ | cauda da distribuição = cauda normal |
| **Henze-Zirkler** | $T_{n,\beta}$ omnibus, p-valor via log-normal | a distribuição conjunta é normal |

In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import metricas_plots

In [ ]:
importlib.reload(metricas_plots)
from metricas_plots import (
    PlotsMetricas, T, F, mardia_test, hz_test, tabela_ocupacao_bpt, plot_ocupacao_bpt
)
p = PlotsMetricas()

In [ ]:
dados = pd.read_csv("dados/ariel_limpo_log10.csv.gz", compression="gzip")
larguras = p.targets[:4]
train, test = train_test_split(
    dados[p.features + larguras], test_size=0.25, random_state=4321
)
test.reset_index().tail()

,index,atflux,atmass,aZflux,aZmass,mass,Av,nii_6584_ew,halpha_ew,oiii_5007_ew,hbeta_ew
31450,72612,9.395308,9.682692,-0.546786,-0.202412,9.869232,0.4500,0.674677,1.019905,0.253822,0.397940
31451,92550,8.150822,9.290086,-0.950915,-1.596153,8.181712,0.4731,0.783618,1.847548,1.702327,1.254886
31452,73774,9.479560,9.851009,-0.467088,-0.354899,10.270185,0.6463,0.879841,1.295655,-0.009217,0.596377
31453,125617,9.511763,9.953186,-0.435457,-0.194487,9.687144,0.5300,0.965672,1.237443,-0.116907,0.514946
31454,74746,9.030630,9.794917,-0.539970,-0.714272,9.035061,0.4205,1.147120,1.663475,0.757320,0.910678


In [ ]:
larguras_lhc = [l+"_median" for l in larguras]
train_lhc = p.lhs_subsample_with_stats(train, p.features, larguras, n=7500, k_neighbors=100)
test_lhc = p.lhs_subsample_with_stats(test, p.features, larguras, n=2500, k_neighbors=100)
test_lhc.tail()

,atflux,atmass,aZflux,aZmass,mass,Av,nii_6584_ew_median,nii_6584_ew_mean,nii_6584_ew_std,halpha_ew_median,...,cov_nii_6584_ew_halpha_ew,cov_nii_6584_ew_oiii_5007_ew,cov_nii_6584_ew_hbeta_ew,cov_halpha_ew_oiii_5007_ew,cov_halpha_ew_hbeta_ew,cov_oiii_5007_ew_hbeta_ew,p_mardia_skew,p_mardia_kurt,p_hz,cell_size
2495,9.041883,9.689113,-0.033978,-0.098239,9.889064,1.1105,0.755910,0.819918,0.251265,1.144214,...,0.047308,0.052755,0.041441,0.043095,0.047670,0.047908,0.000000e+00,0.0,2.654760e-27,100
2496,9.439522,9.682908,-0.547803,-0.570109,9.407381,0.4533,0.625760,0.636769,0.185982,1.065166,...,0.032540,0.024784,0.026882,0.023042,0.028449,0.022262,1.443290e-15,0.0,6.668557e-03,100
2497,9.294197,9.468775,-1.013441,-0.321005,9.381613,0.7100,0.751688,0.773277,0.226197,1.203923,...,0.041385,0.040963,0.038737,0.039446,0.040967,0.040949,0.000000e+00,0.0,3.614584e-20,100
2498,9.358721,9.897927,-0.484938,-0.365912,9.665874,0.3700,0.813848,0.841905,0.174040,1.238034,...,0.025941,0.031307,0.023383,0.025629,0.024153,0.026753,0.000000e+00,0.0,1.486086e-11,100
2499,9.206418,9.662210,-0.669109,-0.937343,9.861502,0.5300,1.077160,1.056480,0.197516,1.506240,...,0.021106,0.026723,0.018143,0.021036,0.017568,0.021096,0.000000e+00,0.0,1.382891e-16,100


In [ ]:
import scipy.linalg as la  # usado no Q-Q plot chi-quadrado (próxima célula)

# ══════════════════════════════════════════════════════════════════════════
# Executar testes em train e train_lhc
# (mardia_test e hz_test vêm de metricas_plots.py, reaproveitados também por
# lhs_subsample_with_stats para as estatísticas por célula do LHC)
# ══════════════════════════════════════════════════════════════════════════
ALPHA    = 0.05
MAX_ROWS = 20000   # limita custo O(n²) do HZ

datasets = {
    "train (5 000 pts)":      train[larguras].dropna().sample(MAX_ROWS, random_state=42).values,
    "train_lhc (2 000 pts)":  train_lhc[larguras_lhc].dropna().values,
}

rows = []
for label, X in datasets.items():
    m = mardia_test(X, alpha=ALPHA)
    h = hz_test(X, alpha=ALPHA)
    rows.append({
        "Dataset":             label,
        "n":                   len(X),
        "b1p (assimetria)":    f"{m['b1p']:.5f}",
        "p (assimetria)":      f"{m['p_skew']:.3e}",
        "Normal (assim)?":     "SIM" if m["normal_skew"] else "NAO",
        "b2p (curtose)":       f"{m['b2p']:.4f}",
        "p (curtose)":         f"{m['p_kurt']:.3e}",
        "Normal (curt)?":      "SIM" if m["normal_kurt"] else "NAO",
        "HZ":                  f"{h['HZ']:.4f}",
        "p (HZ)":              f"{h['p_value']:.3e}",
        "Normal (HZ)?":        "SIM" if h["normal"] else "NAO",
    })

df_testes = pd.DataFrame(rows).set_index("Dataset")
print(f"=== Testes de Multinormalidade  (α = {ALPHA}) ===\n")
display(df_testes.T)


In [ ]:
from scipy.stats import chi2

# ── Q-Q plot chi-quadrado (diagnóstico visual de multinormalidade) ─────────
# Se os dados forem multinormais, as distâncias de Mahalanobis² devem seguir
# uma distribuição χ²(p). O alinhamento com a diagonal confirma a hipótese.
def _chi2_qqplot(ax, X, label):
    X = np.asarray(X, dtype=float)
    n, p = X.shape
    mu  = X.mean(axis=0)
    Xc  = X - mu
    S   = np.cov(Xc.T, bias=False)
    try:
        S_inv = la.inv(S)
    except la.LinAlgError:
        S_inv = la.pinv(S)
    d2 = np.sum(Xc @ S_inv * Xc, axis=1)      # distâncias de Mahalanobis²
    d2_sorted = np.sort(d2)
    probs     = (np.arange(1, n + 1) - 0.5) / n
    chi2_q    = chi2.ppf(probs, df=p)
    ax.scatter(chi2_q, d2_sorted, s=4, alpha=0.35, color="steelblue")
    lim = max(chi2_q.max(), d2_sorted.max())
    ax.plot([0, lim], [0, lim], "r--", lw=1.5, label="y = x")
    ax.set_xlabel(rf"Quantis $\chi^2_{{{p}}}$ teóricos", fontsize=11)
    ax.set_ylabel(r"Mahalanobis$^2$ ordenados", fontsize=11)
    ax.set_title(f"Q-Q multinormal - {label}", fontsize=12)
    ax.legend(fontsize=9)


fig, axes = plt.subplots(1, 2, figsize=(13, 5))
_chi2_qqplot(
    axes[0],
    train[larguras].dropna().sample(2000, random_state=42).values,
    "train (n = 2 000)"
)
_chi2_qqplot(
    axes[1],
    train_lhc[larguras_lhc].dropna().values,
    "train_lhc (n = 2 000)"
)
plt.tight_layout()
plt.savefig("results/multinormalidade_qqplot.png", bbox_inches="tight")
plt.show()

## Gaussian Mixture Models (GMM) como Baseline

Ajuste de GMMs com $k = 1, \ldots, 8$ componentes sobre os dados de treino completos e sobre o subconjunto LHC. O critério BIC seleciona o número ótimo de componentes; se $k^* = 1$ o modelo prefere uma única Gaussiana (consistente com a hipótese de multinormalidade).

In [ ]:
from sklearn.mixture import GaussianMixture

# ── Ajusta GMM com k=1..K_MAX e coleta BIC/AIC ────────────────────────────
def ajustar_gmm(X, k_max=8, n_init=5, random_state=42):
    """Ajusta GMMs com 1..k_max componentes e retorna (ks, bics, aics, modelos)."""
    ks, bics, aics, gms = [], [], [], []
    for k in range(2, k_max + 1):
        gm = GaussianMixture(
            n_components=k, covariance_type="full",
            n_init=n_init, random_state=random_state
        )
        gm.fit(X)
        ks.append(k)
        bics.append(gm.bic(X))
        aics.append(gm.aic(X))
        gms.append(gm)
        print(f"  k={k:2d}  BIC={gm.bic(X):.1f}  AIC={gm.aic(X):.1f}")
    return ks, bics, aics, gms


K_MAX = 20
rng_seed = 4321

# Subamostras para não sobrecarregar o ajuste
X_train_gmm = train[larguras].dropna().sample(7500, random_state=rng_seed).values
X_lhc_gmm   = train_lhc[larguras_lhc].dropna().values

print("=== GMM - train completo (n=7500) ===")
ks_tr,  bics_tr,  aics_tr,  gms_tr  = ajustar_gmm(X_train_gmm, k_max=K_MAX)

print("\n=== GMM - train_lhc ===")
ks_lhc, bics_lhc, aics_lhc, gms_lhc = ajustar_gmm(X_lhc_gmm,   k_max=K_MAX)

# ── Plot BIC/AIC ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ks, bics, aics, title in [
    (axes[0], ks_tr,  bics_tr,  aics_tr,  "GMM - train (n = 5 000)"),
    (axes[1], ks_lhc, bics_lhc, aics_lhc, "GMM - train_lhc (n = 2 000)"),
]:
    ax.plot(ks, bics, "o-", label="BIC", color="navy")
    ax.plot(ks, aics, "s--", label="AIC", color="crimson")
    k_bic = np.argmin(bics) + 1
    k_aic = np.argmin(aics) + 1
    ax.axvline(k_bic, color="navy",   lw=0.8, ls=":", label=f"k* BIC = {k_bic}")
    ax.axvline(k_aic, color="crimson", lw=0.8, ls=":", label=f"k* AIC = {k_aic}")
    ax.set_xlabel("Número de componentes (k)")
    ax.set_ylabel("Critério de informação")
    ax.set_title(title)
    ax.set_xticks(ks)
    ax.legend()

plt.tight_layout()
plt.savefig("results/gmm_bic_aic.png", bbox_inches="tight")
plt.show()

k_best_train = np.argmin(bics_tr)  + 1
k_best_lhc   = np.argmin(bics_lhc) + 1
print(f"\nMelhor k (BIC) - train:    {k_best_train}")
print(f"Melhor k (BIC) - train_lhc: {k_best_lhc}")

In [ ]:
# ── Marginais: dados vs GMM ótimo vs MvN simples ──────────────────────────
best_lhc_model = gms_lhc[k_best_lhc - 1]
n_samp = len(test)
samples_gmm, _ = best_lhc_model.sample(n_samp)
samples_mvn = np.random.default_rng(rng_seed).multivariate_normal(
    X_lhc_gmm.mean(axis=0),
    np.cov(X_lhc_gmm.T),
    size=n_samp
)

def _kl_marginal(p_samples, q_samples, bins):
    edges = np.histogram_bin_edges(np.concatenate([p_samples, q_samples]), bins=bins)
    p_hist = np.histogram(p_samples, bins=edges, density=True)[0]
    q_hist = np.histogram(q_samples, bins=edges, density=True)[0]
    eps = 1e-10
    p_norm = p_hist / (p_hist.sum() + eps)
    q_norm = q_hist / (q_hist.sum() + eps)
    return float(np.sum(p_norm * np.log((p_norm + eps) / (q_norm + eps))))

bins = 60
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (i, l) in zip(axes.flat, enumerate(larguras)):
    kl_gmm = _kl_marginal(X_lhc_gmm[:, i], samples_gmm[:, i], bins=bins)
    kl_mvn = _kl_marginal(X_lhc_gmm[:, i], samples_mvn[:, i], bins=bins)
    ax.hist(X_lhc_gmm[:, i],   bins=bins, density=True, alpha=0.333, label="train_lhc", color="darkblue")
    ax.hist(samples_gmm[:, i],  bins=bins, density=True, alpha=0.333, label=f"GMM (k={k_best_lhc})", color="darkred")
    ax.hist(samples_mvn[:, i],  bins=bins, density=True, alpha=0.333, label="MvN", color="teal")
    ax.set_xlabel(l)
    ax.set_title(r"$D_{KL}$ = %.4f (GMM) / %.4f (MvN)" % (kl_gmm, kl_mvn))
    ax.legend(fontsize=8)
axes[0, 0].set_ylabel("Densidade")
axes[1, 0].set_ylabel("Densidade")
plt.suptitle(f"Marginals: LHC_means vs GMM (k={k_best_lhc}) vs Multivariate Normal")
plt.tight_layout()
plt.savefig("results/gmm_vs_mvn_marginais.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Diagramas BPT e ocupação das regiões: train_lhc vs MvN vs GMM ────────
def bpt_frame(X):
    df = pd.DataFrame(X, columns=larguras)
    df[T.nii_ha.value] = df[T.nii.value] - df[T.ha.value]
    df[T.oiii_hb.value] = df[T.oiii.value] - df[T.hb.value]
    return df


df_lhc = bpt_frame(X_lhc_gmm)
df_mvn = bpt_frame(samples_mvn)
df_gmm = bpt_frame(samples_gmm)

known_x_bpt = df_lhc[T.nii_ha.value]
known_y_bpt = df_lhc[T.oiii_hb.value]

train_bpt_x = train[T.nii.value] - train[T.ha.value]
train_bpt_y = train[T.oiii.value] - train[T.hb.value]

# Diagramas BPT: cada conjunto amostrado vs train_lhc (Validation Set)
for nome, df_s in [("MvN", df_mvn), (f"GMM_k{k_best_lhc}", df_gmm)]:
    p.show_bpt(
        df_s,
        title=f"Amostras {nome} vs train_lhc",
        densities=True,
        show=False,
        known_x=known_x_bpt,
        known_y=known_y_bpt,
        background_x=train_bpt_x,
        background_y=train_bpt_y,
    )
    plt.savefig(f"results/gmm_bpt_{nome}.png", bbox_inches="tight")
    plt.show()

# Tabela + gráfico de ocupação das regiões do BPT (SF / Composite / AGN)
df_ocupacao = tabela_ocupacao_bpt({
    "train_lhc": (df_lhc[T.nii_ha.value], df_lhc[T.oiii_hb.value]),
    "MvN": (df_mvn[T.nii_ha.value], df_mvn[T.oiii_hb.value]),
    f"GMM (k={k_best_lhc})": (df_gmm[T.nii_ha.value], df_gmm[T.oiii_hb.value]),
})
df_ocupacao.to_csv("results/gmm_bpt_ocupacao.csv")

plot_ocupacao_bpt(df_ocupacao)
plt.tight_layout()
plt.savefig("results/gmm_bpt_ocupacao.png", bbox_inches="tight")
plt.show()

df_ocupacao